## Searching Technical Documents with Text vectors and cosine similariy
**EGR 444 Peer-Led ML Workshop** 45 minutes - Google Colab, CPU only, no training

**README Section EGR 444 ML Workshop**
*AI Used for VOCAB AND COLAB text effects only including (where to seperate cells, how to use **these** and how to create tables)*

### **Schedule**
What are some limitations of these methods?
| Minutes | Objective | Notebook |
|---|---|---|
| 0-5 | Project needs; show the finished side by side results | Step 5 Output. |
| 5-10 | Main idea is text to vectors to cosine angle. Everyone runs step 1. | #Checkpoint 1 |
|10-20 | TF-IDF build and search. Discover the wrong #1 result and why | Checkpoints #2-4 |
| 20-30 | Embeddings built and searched. Then side by side comparison with embeddings and the cosine | Checkpoints #5-6 |
| 30-35 | Participant task | Step #6 |
| 35-40 | Handoff: when to use it and when to not to. Next steps and questions. | Step #7
| 40-45 | Troubleshooting and spare time


## 1. **The Project Needs**
A technician has a DC motor that won't start and a long equipment manual. Instead of scrolling through the PDF or flipping through pages they type **Why isnt my DC motor turning on?** and get the most relevent troubleshooting section returned.

## 2. **Task that **YOU** Need to complete!**
A search within a colab notebook that ranks 30 manual sections by **cosine similarity** based on a user imposed question. The tool then returns **3 matches based on a score** that measures relativity to questions. Our plan is to build it twice using **TR-IDF** first then **Cosine Similarity** after that. Then you'll run your own question through both, and explain why the results are different or the same.

##3. **Files You'll need**
| File | What it is |
|---|---|
| `workshop_notebook.ipynb` | **Start here.** The tutorial notebook, with expected outputs saved in it |
| `data/dc_motor_manual.csv` | 30 short manual sections (`section_id`, `title`, `text`) |
| `README.md` | This guide: setup, steps, expected results, troubleshooting |

## 4. **Setup**
**3-5 minutes**

A. open `workshop-notebook.ipynb` (**file -> Upload notebook**, or use the GitHub link).

B. The runtime can stay on **CPU**. **GPU** not required!

C. Run the **Step 0** cell. This installs `sentence-transformers` (Don't worry about red dependency from pip, that's normal.)

D. Run the **Step 1** cell. This downloads the CSV from `DATA_URL`. If it can't upload `dc_motor_manual.csv` with the folder icon on the left toolbar. Then run the cell again.

Requirements: Colab doesn't have **sentence-transformers**. But it does have `pandas`, `numpy`, `scikit-learn`,

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
%pip -q install sentence-transformers

In [7]:
import pandas as pd
from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

## 5. **Tutorial and Checkpoints**
| Step | What you do | `df.shape` -> `(30,315)` |
|---|---|---|
| 1 | Load the CSV | `df.shape` -> `(30,3)` |
| 2 | TF-IDF vectors | `X_tfidf.shape` -> `(30,315)` |
| 3a | Make question into vector | question `(1, 315)`, scores `(30,)` |
| 3b | TF_IDF top 3 answers *"why isn't my motor turning on?"* |`0.340 Turning the shaft during installation` · `0.171 Motor runs in the wrong direction` · `0.122 Insulation resistance test` (**wrong section on top!**) |
| 3c | Look at the words TF-IDF finds | `Isn't, dc, motor, turning: it matched on "turning" and "dc", not on the meaning |
| 4a | Embeddings with `all-MiniM-L6-v2` | `x_emb.shape` -> `(30,384)` (Section * embedding size) |
| 4b | EmbeddingTop 3, same question | **Motor sails to start** at or near #1 |
| 5 | `Compare()` both medthods side by side | `Two ranked lists |
| 6 | **Your turn:** Your own question + explanation |Check section #6|

**Reasonable score ranges** (the two methods use different scales, so compare rankings, not raw numbers):
| | Strong match | Bad match |
|---|---|---|
| TF-IDF | ~0.3 and up | Exactly 0,0 when no words are present |
| Embeddings | ~0.5 and up | ~0.0 - 0.2, rarely 0 |

Scores can differ in the 3rd decimal place. Don't worry about it.

In [8]:
#Load Dataset
files.upload()
df = pd.read_csv("dc_motor_manual.csv")
documents = (df["title"] + " " + df["text"]).tolist()

Saving dc_motor_manual.csv to dc_motor_manual.csv


In [9]:
# Explore data
print(df.shape)


(30, 3)




```
# This is formatted as code
```

## 6. **Your Task**
1. Change the `MY_Question` in the code to your own troubleshooting question. Describe a symptom **without** using the manuals words. *"it smells like smoke"* or *"The motor keeps speeding up and slowing down*"
2. Run the `compare(MY_QUESTION)`
3. In the notebook, write down: Did both methods pick the same #1? if not, **which words** explain the TF-IDF result? Which one would you trust?

**How to check answer:** a good explanation points to specific word matches or mismatches, such as 'Turning' in the installation section, while the embedded section

In [10]:
#Enter a question
question = "Why does my machine smell like smoke?"

In [18]:
#TF-IDF & Cosine Similarity Search
tfidf = TfidfVectorizer(ngram_range=(1,1))
vectors = tfidf.fit_transform(documents)
scores_tfidf = cosine_similarity(tfidf.transform([question]), vectors)[0]


In [19]:
# Sentence embedded search
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(documents)
query = model.encode([question])
scores_similarity = cosine_similarity(query, embeddings)[0]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
# Top 3 Results
for method, scores in [("TF-IDF", scores_tfidf),
                       ("Cosine Similarity", scores_similarity)]:
    print(f"\n{method} - Top 3 Results")

    for i in scores.argsort()[-3:][::-1]:
        print(f"\n{df.iloc[i]['title']} | Similarity: {scores[i]:.3f}")
        print(df.iloc[i]["text"])


TF-IDF - Top 3 Results

Burning smell | Similarity: 0.317
A burning smell usually comes from overheated windings or insulation. Stop the motor immediately, disconnect power, and perform an insulation resistance test before restarting.

Motor fails to start | Similarity: 0.116
If the motor does not start when the run command is given, check that supply voltage is present at the drive input terminals. Inspect the fuse and reset the circuit breaker if it has tripped.

No rotation with power applied | Similarity: 0.108
When power is present but the shaft does not rotate, the load may be jammed or the brake may still be engaged. Release the brake and try rotating the shaft by hand.

Cosine Similarity - Top 3 Results

Burning smell | Similarity: 0.641
A burning smell usually comes from overheated windings or insulation. Stop the motor immediately, disconnect power, and perform an insulation resistance test before restarting.

Commutator cleaning | Similarity: 0.400
A dirty or burned commuta

## 7. **Schedule**
What are some limitations of these methods?
| Time (Min) | Activity| Notebook |
|---|---|---|
| 0-5 | Project needs; show the finished side by side results | Step 5 Output. |
| 5-10 | Main idea is text to vectors to cosine angle. Everyone runs step 1. | #Checkpoint 1 |
|10-20 | TF-IDF build and search. Discover the wrong #1 result and why | Checkpoints #2-4 |
| 20-30 | Embeddings built and searched. Then side by side comparison with embeddings and the cosine | Checkpoints #5-6 |
| 30-35 | Participant task | Step #6 |
| 35-40 | Handoff: when to use it and when to not to. Next steps and questions. | Step #7
| 40-45 | Troubleshooting and spare time

In [ ]:
# Top 3 Results
for method, scores in [("TF-IDF", scores_tfidf),
                       ("Cosine Similarity", scores_similarity)]:
    print(f"\n{method} - Top 3 Results")

    for i in scores.argsort()[-3:][::-1]:
        print(f"\n{df.iloc[i]['title']} | Similarity: {scores[i]:.3f}")
        print(df.iloc[i]["text"])


TF-IDF - Top 3 Results

Burning smell | Similarity: 0.317
A burning smell usually comes from overheated windings or insulation. Stop the motor immediately, disconnect power, and perform an insulation resistance test before restarting.

Motor fails to start | Similarity: 0.116
If the motor does not start when the run command is given, check that supply voltage is present at the drive input terminals. Inspect the fuse and reset the circuit breaker if it has tripped.

No rotation with power applied | Similarity: 0.108
When power is present but the shaft does not rotate, the load may be jammed or the brake may still be engaged. Release the brake and try rotating the shaft by hand.

Cosine Similarity - Top 3 Results

Burning smell | Similarity: 0.641
A burning smell usually comes from overheated windings or insulation. Stop the motor immediately, disconnect power, and perform an insulation resistance test before restarting.

Commutator cleaning | Similarity: 0.400
A dirty or burned commuta

## 8. Troubleshooting
| Error | Most Likely cause | How to Fix |
|---|---|---|
| `FileNotFoundError: dc_motor_manual.csv` | CSV not downloaded or uploaded | Upload it with the 📁 icon on the left toolbar. Make sure it downloads fully in the Load Dataset column
| CSV loads but `df.shape` is wrong or it's full of HTML | `DATA_URL` points to the GitHub *page*, not the *raw* file | Use the URL from the **Raw** button (`raw.githubusercontent.com/...`). Delete the bad file with `!rm dc_motor_manual.csv` and rerun |
| `ModuleNotFoundError: No module named 'sentence_transformers'` | Step 0 not run, or the runtime restarted | Rerun Step 0 |
| `NameError: name 'tfidf' / 'model' / 'X_emb' is not defined` | Cells run out of order, or the runtime reset | **Runtime → Run before** on the current cell |
| `ValueError: Incompatible dimension for X and Y matrices` | Question and sections are in different vector spaces (e.g. `fit_transform` on the question, or a TF-IDF question vs. `X_emb`) | Use `tfidf.transform([q])` with `X_tfidf`, and `model.encode([q])` with `X_emb` |
| TF-IDF top 3 are all `0.000` | None of your question's words are in the manual's vocabulary | That's the expected keyword-search weakness, so explain it in your answer. Check with `analyzer(MY_QUESTION)` |
| Model download hangs or shows an HTTP/Hugging Face error | Network hiccup or rate limit | Wait 30 seconds and rerun the cell. If it still fails, **Runtime → Restart session** and rerun from the top |
| "Runtime disconnected" | Colab idle timeout | Reconnect, then **Runtime → Run all** (about 1 minute) |
| Red pip warnings about dependency conflicts in Step 0 | Colab's preinstalled packages | Safe to ignore if it prints `Setup done` |

## 9. Handoff: when to use this
- **Useful when:** any team has a pile of documents (manuals, SOPs, tickets, papers) and people need the right passage quickly.
- **TF-IDF:** instant, no download, explainable, and strong on exact codes and terms. Weak on synonyms.
- **Cosine Similarity:** handles paraphrases and synonyms. Needs a small model download and can blur exact identifiers. Many systems combine both ("hybrid search").
- **Key limitation:** it **finds** relevant text but does **not answer** the question. In a full RAG system, the top-k sections go to an LLM that writes the answer.
- **Adapt it:** swap in your own CSV of text chunks (one paragraph per row) and keep the code as is.

## **Documentation**
- scikit-learn TfidfVectorizer: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- scikit-learn cosine_similarity: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html
- sentence-transformers: https://sbert.net
- all-MiniLM-L6-v2: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

**Credits:** The manual text is a simplified, workshop-written example and is not from a real product manual. The model is `all-MiniLM-L6-v2` by the sentence-transformers project (Apache 2.0).

In [ ]:
# Top 3 Results
for method, scores in [("TF-IDF", scores_tfidf),
                       ("Cosine Similarity", scores_similarity)]:
    print(f"\n{method} - Top 3 Results")

    for i in scores.argsort()[-3:][::-1]:
        print(f"\n{df.iloc[i]['title']} | Similarity: {scores[i]:.3f}")
        print(df.iloc[i]["text"])


TF-IDF - Top 3 Results

Burning smell | Similarity: 0.317
A burning smell usually comes from overheated windings or insulation. Stop the motor immediately, disconnect power, and perform an insulation resistance test before restarting.

Motor fails to start | Similarity: 0.116
If the motor does not start when the run command is given, check that supply voltage is present at the drive input terminals. Inspect the fuse and reset the circuit breaker if it has tripped.

No rotation with power applied | Similarity: 0.108
When power is present but the shaft does not rotate, the load may be jammed or the brake may still be engaged. Release the brake and try rotating the shaft by hand.

Cosine Similarity - Top 3 Results

Burning smell | Similarity: 0.641
A burning smell usually comes from overheated windings or insulation. Stop the motor immediately, disconnect power, and perform an insulation resistance test before restarting.

Commutator cleaning | Similarity: 0.400
A dirty or burned commuta

## 10. Contribution note (for instructor)
| Member | Preparation | Workshop role |
|---|---|---|
| *Nicholas Doran* | dataset, notebook TF-IDF section | leads 10–20 min, floats 20–40 |
| *Connor Valdez* | Embeddings, Cosine Similarity, created slides, notebook|leads 10–20 min, floats 20–40 |

